In [1]:
from pathlib import Path
import os
import sys


root_folder = next(
    (
        parent
        for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (parent / "rag_project" / "evaluation" / "config.py").is_file()
    ),
    None,
)
if root_folder is None:
    raise FileNotFoundError(
        "Could not find the project root containing rag_project/evaluation/config.py"
    )
root_folder = str(root_folder)
if root_folder not in sys.path:
    sys.path.insert(0, root_folder)
root_folder

'/Users/harendrakumar/Documents/pythonProjects'

In [2]:
os.chdir(root_folder)
os.getcwd()

'/Users/harendrakumar/Documents/pythonProjects'

In [3]:
from rag_project.evaluation.config import RetrievalConfig
from rag_project.evaluation.results import RetrievalResult

from rag_project.evaluation.metrics import (
    calculate_recall_at_k, 
    calculate_precision_at_k,
    calculate_f1
    )

from rag_project.evaluation.retrieval_dataset import EVALUATION_DATASET
from rag_project.evaluate_retrieval import (
    build_retriever,
    get_relevant_chunk_ids
    )

ModuleNotFoundError: No module named 'evaluation'

In [8]:
for item in EVALUATION_DATASET:
    print(item)

{'question': 'What is the DOM?', 'relevant_pages': [8]}
{'question': 'What is the difference between the DOM and HTML source code?', 'relevant_pages': [11, 12, 14]}
{'question': 'How can JavaScript modify the DOM?', 'relevant_pages': [12, 13]}
{'question': 'Why can the DOM differ from the original HTML source?', 'relevant_pages': [12, 14]}


In [10]:
class RetrievalExperiment:
    def run(self, config: RetrievalConfig) -> RetrievalResult:
        retriever, chunks = build_retriever(
            chunk_size=config.chunk_size,
            chunk_overlap = config.chunk_overlap
        )
        recalls = []
        precisions= []

        for item in EVALUATION_DATASET:
            relevant_chunk_ids = get_relevant_chunk_ids(
                chunks=chunks,
                relevant_chunks=item["relevant_pages"]
            )

            results = retriever.retrieve(question= item["question"],
                                         top_k = config.top_k
                                        )
            recall = calculate_recall_at_k(retrieved_chunks=results,
                                           relevant_chunk_ids=relevant_chunk_ids
                                           )
            precision = calculate_precision_at_k(retrieved_chunks=results,
                                                 relevant_chunk_ids=relevant_chunk_ids
                                                )
            recalls.append(recall)
            precisions.append(precision)

        average_recall = sum(recalls) / len(recalls)
        average_precision = sum(precisions) / len(precisions)

        f1 = calculate_f1(
            precision=average_precision,
            recall = average_recall
        )

        return RetrievalResult(
            chunk_size=config.chunk_size,
            chunk_oervalap=config.chunk_overlap,
            top_k = config.top_k,
            total_chunks=len(chunks),
            recall = average_recall,
            precision = average_precision,
            f1 = f1
        )
            

        

In [11]:
# run_experiment.py
from rag_project.evaluation.experiment import RetrievalExperiment
from rag_project.evaluation.config import RetrievalConfig

def main():
    experiment = RetrievalExperiment()
    results = []

    chunking_configs = [
        (500, 50),
        (800, 100),
        (1200, 200)
        ]

    top_k_values = [1, 3, 5, 10]

    for chunk_size, chunk_overlap in chunking_configs:
        for top_k in top_k_values:
            config = RetrievalConfig(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                top_k=top_k
            )
            print(
                f"\nRunning: "
                f"chunk_size={chunk_size} "
                f"chunk_overlap={chunk_overlap} "
                f"top_k={top_k}"
            )

            result = experiment.run(config)

            results.append(result)

    print("\n" + "=" * 90)
    print("Retrieval Experiment Results: ")
    print("="* 90)

    print(
        f"{'Chunk':>8} "
        f"{'Overlap':>8} "
        f"{'K':>5} "
        f"{'Chunks':>8} "
        f"{'Recall':>10} "
        f"{'Precision':>10} "
        f"{'F1':>10}"
    )

    for result in results:
        print(
            f"{result.chunk_size:>8} "
            f"{result.chunk_overlap:>8} "
            f"{result.top_k:>5} "
            f"{result.total_chunks:>8} "
            f"{result.recall:>10.3f} "
            f"{result.precision:>10.3f} "
            f"{result.f1:>10.3f}"
        )

    if __name__ == "__main__":
        main()




In [3]:
import speedtest

st = speedtest.Speedtest()

download = st.download() / 1_000_000
upload = st.upload() / 1_000_000
ping = st.results.ping

print(f"Download speed: {download:.2f} Mbps")
print(f"Upload speed: {upload:.2f} Mbps")
print(f"Ping: {ping} ms")

Download speed: 98.93 Mbps
Upload speed: 94.16 Mbps
Ping: 66.98 ms
